# 07 — Monitoring & Data Drift Analysis

## Objectif

Mettre en place un système de monitoring pour le modèle de scoring en production :
1. **Simuler des données de production** (prédictions sur le test set)
2. **Analyser les métriques opérationnelles** (distribution des scores, latence, décisions)
3. **Détecter la dérive des données (data drift)** avec Evidently AI
4. **Identifier les points de vigilance** pour le maintien du modèle

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone, timedelta

import warnings
warnings.filterwarnings('ignore')

print("Imports OK")

## 1. Simulation des données de production

En l'absence de trafic réel, on simule des requêtes API en passant le **test set** 
dans le modèle. Chaque prédiction est loguée au format JSONL, exactement comme 
le ferait l'API Gradio en production.

In [ ]:
# Charger le modele et la config
model = joblib.load('../model/model.pkl')
with open('../model/config.json') as f:
    config = json.load(f)

THRESHOLD = config['threshold']
FEATURE_NAMES = config['feature_names']

# Charger les donnees de test (simulation de production)
df_test = pd.read_csv('../data/test_preprocessed.csv')
df_test.columns = df_test.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)

print(f"Modele charge : {type(model).__name__}")
print(f"Seuil : {THRESHOLD:.2f}")
print(f"Donnees de test : {df_test.shape}")

In [ ]:
# Simuler les predictions de production avec mesure de latence
X_prod = df_test[FEATURE_NAMES]
client_ids = df_test['SK_ID_CURR'].values

# Predictions en batch (rapide)
probas = model.predict_proba(X_prod)[:, 1]

# Mesurer la latence sur un echantillon de 500 predictions individuelles
n_latency_samples = 500
latency_samples = []
idx_sample = np.random.choice(len(X_prod), n_latency_samples, replace=False)
for i in idx_sample:
    start = time.perf_counter()
    _ = model.predict_proba(X_prod.iloc[[i]])
    latency_samples.append((time.perf_counter() - start) * 1000)

# Construire le log complet
logs = []
for i in range(len(probas)):
    decision = "REFUSE" if probas[i] >= THRESHOLD else "ACCORDE"
    
    # Timestamps simules sur 7 jours
    fake_ts = datetime(2026, 3, 23, tzinfo=timezone.utc) + timedelta(
        seconds=int(np.random.randint(0, 7 * 86400))
    )
    
    # Assigner une latence mesuree ou interpolee
    inference_ms = latency_samples[i % n_latency_samples] + np.random.normal(0, 0.5)
    
    logs.append({
        'timestamp': fake_ts.isoformat(),
        'client_id': int(client_ids[i]),
        'probability': round(float(probas[i]), 6),
        'decision': decision,
        'threshold': THRESHOLD,
        'inference_time_ms': round(max(0.1, inference_ms), 2),
    })

df_logs = pd.DataFrame(logs)
df_logs['timestamp'] = pd.to_datetime(df_logs['timestamp'])

# Sauvegarder en JSONL (meme format que l'API)
import os
os.makedirs('../logs', exist_ok=True)
with open('../logs/predictions.jsonl', 'w') as f:
    for log in logs:
        f.write(json.dumps(log) + '\n')

print(f"Simulation terminee : {len(df_logs)} predictions")
print(f"Decisions : {df_logs['decision'].value_counts().to_dict()}")
print(f"Logs sauvegardes dans ../logs/predictions.jsonl")

## 2. Dashboard de métriques opérationnelles

Visualisation des métriques clés que l'on surveille en production :
- Distribution des scores prédits
- Répartition des décisions (ACCORDÉ / REFUSÉ)
- Temps d'inférence (latence)
- Volume de requêtes dans le temps

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution des scores
axes[0, 0].hist(df_logs['probability'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Seuil = {THRESHOLD:.2f}')
axes[0, 0].set_title('Distribution des scores de probabilite')
axes[0, 0].set_xlabel('Probabilite de defaut')
axes[0, 0].set_ylabel('Nombre de clients')
axes[0, 0].legend()

# 2. Repartition des decisions
decision_counts = df_logs['decision'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0, 1].pie(decision_counts.values, labels=decision_counts.index, 
               autopct='%1.1f%%', colors=colors, startangle=90)
axes[0, 1].set_title('Repartition des decisions')

# 3. Distribution de la latence
axes[1, 0].hist(df_logs['inference_time_ms'], bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1, 0].axvline(df_logs['inference_time_ms'].median(), color='red', linestyle='--', 
                    label=f"Mediane = {df_logs['inference_time_ms'].median():.1f} ms")
axes[1, 0].axvline(df_logs['inference_time_ms'].quantile(0.95), color='darkred', linestyle=':', 
                    label=f"P95 = {df_logs['inference_time_ms'].quantile(0.95):.1f} ms")
axes[1, 0].set_title("Distribution du temps d'inference")
axes[1, 0].set_xlabel('Temps (ms)')
axes[1, 0].set_ylabel('Nombre de requetes')
axes[1, 0].legend()

# 4. Volume de requetes par jour
df_logs_sorted = df_logs.sort_values('timestamp')
daily_counts = df_logs_sorted.set_index('timestamp').resample('D').size()
axes[1, 1].bar(daily_counts.index.strftime('%d/%m'), daily_counts.values, color='mediumpurple')
axes[1, 1].set_title('Volume de requetes par jour')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Nombre de requetes')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('Dashboard de Monitoring — Production', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../captures/monitoring_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nStatistiques de latence :")
print(f"  Moyenne : {df_logs['inference_time_ms'].mean():.2f} ms")
print(f"  Mediane : {df_logs['inference_time_ms'].median():.2f} ms")
print(f"  P95     : {df_logs['inference_time_ms'].quantile(0.95):.2f} ms")
print(f"  P99     : {df_logs['inference_time_ms'].quantile(0.99):.2f} ms")

## 3. Détection de Data Drift avec Evidently AI

### Qu'est-ce que le data drift ?

Le **data drift** survient quand la distribution des données en production diverge 
de celle des données d'entraînement. Cela peut dégrader les performances du modèle 
sans que les métriques de base (AUC, accuracy) ne le détectent immédiatement.

### Approche
- **Référence** : échantillon du jeu d'entraînement (`reference_data.csv`)
- **Production** : données du test set scorées par le modèle
- **Test statistique** : Kolmogorov-Smirnov pour les variables numériques, chi² pour les catégorielles

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset

# Donnees de reference (echantillon du train)
df_reference = pd.read_csv('../data/reference_data.csv')

# Donnees de production (test set sans SK_ID_CURR)
df_production = df_test.drop(columns=['SK_ID_CURR'])

# Les colonnes doivent correspondre
common_cols = [c for c in df_reference.columns if c in df_production.columns]
df_reference = df_reference[common_cols]
df_production = df_production[common_cols]

print(f"Reference : {df_reference.shape}")
print(f"Production : {df_production.shape}")
print(f"Colonnes communes : {len(common_cols)}")

### 3.1 Rapport de Data Drift (sans drift artificiel)

On compare les distributions entre les données d'entraînement et de production 
**sans modification**. Normalement, le drift devrait être faible puisque les données 
proviennent de la même source.

In [ ]:
# Rapport de drift sans perturbation
drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(reference_data=df_reference, current_data=df_production.sample(5000, random_state=42))

# Sauvegarder le rapport HTML
os.makedirs('../reports', exist_ok=True)
drift_report.save_html('../reports/data_drift_report_no_drift.html')
print("Rapport sauvegarde : ../reports/data_drift_report_no_drift.html")

# Extraire les resultats
drift_results = drift_report.as_dict()
dataset_drift = drift_results['metrics'][0]['result']
print(f"\nDataset Drift detecte : {dataset_drift['dataset_drift']}")
print(f"Part de features avec drift : {dataset_drift['share_of_drifted_columns']:.2%}")
print(f"Nombre de features avec drift : {dataset_drift['number_of_drifted_columns']} / {dataset_drift['number_of_columns']}")

### 3.2 Simulation de drift contrôlé

Pour démontrer la capacité de détection, on introduit un **drift artificiel** 
en perturbant certaines features clés :
- `AMT_INCOME_TOTAL` : revenus multipliés par 1.5 (inflation salariale)
- `DAYS_BIRTH` : clients plus jeunes (-2000 jours)
- `EXT_SOURCE_1/2/3` : scores externes dégradés (-0.2)

Ce scénario simule un changement de population de clients demandant un crédit.

In [ ]:
# Creer un dataset avec drift artificiel
df_drifted = df_production.sample(5000, random_state=42).copy()

# Perturber les features cles
if 'AMT_INCOME_TOTAL' in df_drifted.columns:
    df_drifted['AMT_INCOME_TOTAL'] *= 1.5  # Inflation salariale

if 'DAYS_BIRTH' in df_drifted.columns:
    df_drifted['DAYS_BIRTH'] += 2000  # Clients plus jeunes

for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    if col in df_drifted.columns:
        df_drifted[col] = (df_drifted[col] - 0.2).clip(0, 1)  # Scores degrades

print("Features perturbees :")
for col in ['AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    if col in df_drifted.columns:
        ref_mean = df_reference[col].mean()
        drift_mean = df_drifted[col].mean()
        print(f"  {col}: ref={ref_mean:.2f} → drift={drift_mean:.2f}")

In [ ]:
# Rapport de drift avec perturbation
drift_report_drifted = Report(metrics=[DataDriftPreset()])
drift_report_drifted.run(reference_data=df_reference, current_data=df_drifted)

drift_report_drifted.save_html('../reports/data_drift_report_with_drift.html')
print("Rapport sauvegarde : ../reports/data_drift_report_with_drift.html")

drift_results_drifted = drift_report_drifted.as_dict()
dataset_drift_d = drift_results_drifted['metrics'][0]['result']
print(f"\nDataset Drift detecte : {dataset_drift_d['dataset_drift']}")
print(f"Part de features avec drift : {dataset_drift_d['share_of_drifted_columns']:.2%}")
print(f"Nombre de features avec drift : {dataset_drift_d['number_of_drifted_columns']} / {dataset_drift_d['number_of_columns']}")

# Lister les features qui ont drifte
drifted_cols = []
for col_name, col_data in dataset_drift_d['drift_by_columns'].items():
    if col_data['drift_detected']:
        drifted_cols.append((col_name, col_data['drift_score']))

drifted_cols.sort(key=lambda x: x[1])
print(f"\nFeatures avec drift detecte :")
for col, score in drifted_cols:
    print(f"  {col}: p-value = {score:.6f}")

### 3.3 Comparaison visuelle : avec et sans drift

In [ ]:
# Visualiser le drift sur les features cles
drift_features = ['AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
drift_features = [f for f in drift_features if f in df_reference.columns]

fig, axes = plt.subplots(1, len(drift_features), figsize=(5 * len(drift_features), 4))

for i, col in enumerate(drift_features):
    ax = axes[i] if len(drift_features) > 1 else axes
    ax.hist(df_reference[col].dropna(), bins=30, alpha=0.5, label='Reference (train)', density=True)
    ax.hist(df_drifted[col].dropna(), bins=30, alpha=0.5, label='Production (drift)', density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Comparaison des distributions : Reference vs Production (avec drift)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../captures/drift_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Rapport de qualité des données

In [ ]:
# Rapport de qualite des donnees de production
quality_report = Report(metrics=[DataQualityPreset()])
quality_report.run(reference_data=df_reference, current_data=df_production.sample(5000, random_state=42))

quality_report.save_html('../reports/data_quality_report.html')
print("Rapport de qualite sauvegarde : ../reports/data_quality_report.html")

## 5. Conclusions et points de vigilance

### Résultats du monitoring

**Sans drift artificiel :**
- Le drift entre train et test est attendu comme faible (mêmes processus de collecte)
- Un léger drift naturel peut exister sur certaines features

**Avec drift artificiel :**
- Evidently détecte correctement les features perturbées
- Les features **EXT_SOURCE** sont les plus critiques (top 3 SHAP) : un drift sur ces features 
  impacte directement la qualité des prédictions

### Points de vigilance en production

1. **EXT_SOURCE_1/2/3** : ces features dominent le modèle. Si le fournisseur externe change 
   sa méthode de calcul, le modèle sera fortement impacté → **alerter dès qu'un drift est détecté**
2. **AMT_INCOME_TOTAL** : sensible à l'inflation et aux changements de politique salariale
3. **DAYS_BIRTH** : un changement de cible démographique (ex: nouveaux produits pour jeunes) 
   déclenchera un drift → **recalibrer le seuil ou ré-entraîner**
4. **Latence** : surveiller le P95 ; si > 100ms, investiguer (charge serveur, taille des données)

### Recommandations
- Exécuter le rapport Evidently **hebdomadairement** sur les dernières prédictions
- Mettre en place une **alerte** si > 20% des features dérivent
- Planifier un **ré-entraînement** trimestriel ou dès qu'un drift significatif est confirmé
- Conserver les logs de prédiction pour analyse rétrospective

## 5. Conclusions et points de vigilance

### Résultats du monitoring

1. **Sans drift artificiel** : le data drift entre train et test est faible, ce qui est attendu puisque les données proviennent de la même source Home Credit.

2. **Avec drift artificiel** : Evidently détecte correctement les features perturbées (AMT_INCOME_TOTAL, DAYS_BIRTH, EXT_SOURCE_*). Cela valide que le pipeline de détection fonctionne.

### Points de vigilance pour la production

| Risque | Feature(s) concernée(s) | Action recommandée |
|--------|------------------------|-------------------|
| **Drift des scores externes** | EXT_SOURCE_1/2/3 | Monitorer en priorité — ces features ont le plus d'impact SHAP |
| **Changement de population** | DAYS_BIRTH, AMT_INCOME_TOTAL | Alerter si la moyenne s'écarte de >10% de la référence |
| **Latence anormale** | inference_time_ms | Alerter si P95 > 100ms |
| **Taux de refus anormal** | decision | Alerter si le ratio REFUSÉ dépasse 30% |

### Recommandations

- **Fréquence de monitoring** : analyser le drift toutes les semaines (ou tous les 1000 scores)
- **Seuil d'alerte** : si >20% des features montrent du drift significatif → investiguer
- **Réentraînement** : si l'AUC en production descend sous 0.72 → relancer le pipeline d'entraînement
- **RGPD** : ne pas stocker les données personnelles dans les logs, uniquement l'ID client et le score